# 2 — Training: the frozen-trunk box head on Brackish

Backbone verification, a throughput probe, the frozen-trunk go/no-go run (8 epochs × 1 tile/frame) and the linear-probe diagnostic.

> Run **`1_reformat.ipynb` first** (it writes the CFD manifest to Drive). The Brackish frames live on the VM disk at `/content/data/brackish`; if this notebook lands on a fresh runtime, cell 1b re-creates them. Everything else (weights, run dirs, results) is on Drive and persists. On an A100 the package picks bfloat16 automatically; on a T4 it picks fp16 + GradScaler.

## 1 · Drive, paths, code

Mounts Drive, fixes the four paths every cell below uses, clones the branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1b · Make sure the Brackish slice is on this VM

Colab gives each notebook its own runtime, so the frames fetched by `1_reformat.ipynb` are not here unless you attached this notebook to that same session. This cell re-creates the slice only if it is missing (metadata 47 MB, ~14.7k frames from the LILA GCS mirror; ~4–5 min).

In [ ]:
# Idempotent: skip if 1_reformat already populated this runtime.
have = os.path.exists(f'{DATA}/val/annotations.json') and os.path.isdir(f'{DATA}/val/images') and len(os.listdir(f'{DATA}/val/images')) > 0
if not have:
    META = f'{CFD}/community_fish_detection_dataset.json.zip'
    if not os.path.exists(META):
        !wget -q -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources brackish_dataset --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    !python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
!ls {DATA}/train/images | wc -l; ls {DATA}/val/images | wc -l

## 2 · Machine

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, torch
print('vCPUs:', os.cpu_count(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
from cropcounter.dinov3_pyramid import amp_dtype
_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('compute capability:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
      '| AMP dtype the package will use:', amp_dtype(_dev), '(T4 = sm_75 -> float16 + GradScaler; Ampere+ -> bfloat16)')
!df -h /content | tail -1

## 3 · Backbone weights + metric-reproduction check

The DINOv3 checkpoint is Meta-gated; the file on Drive is the ungated timm re-host converted to Meta's parameter names (see vault memory `reference_dinov3_gated_weights`). A strict load only proves the *shapes* match, so we verify by **reproducing a metric**: the shipped wheat decoder `weights/decoder_best.pt` (tracked in git) on the six committed example val images must reproduce the committed counts in `examples/demo/data/val/expected_counts.csv` (±1 tolerance). The committed counts were produced on MPS in **fp32**, so the gate runs the six images with autocast **disabled** (`predict_prob(..., amp=False)`) — like for like. The same six are then run again under whatever `amp_dtype` picks for this GPU and the deltas are *printed*: on a T4 (fp16) the densest image comes back 89 against the committed 92, which is a precision difference, not a backbone difference, and must not fail a backbone check.

In [ ]:
import shutil, os, csv, torch
from pathlib import Path
os.makedirs(f'{REPO}/weights', exist_ok=True)
src = f'{DRIVE}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
dst = f'{REPO}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
if not os.path.exists(dst):
    shutil.copy(src, dst)   # one-off 350 MB Drive read; local SSD from here on
print(os.path.getsize(dst) / 1e6, 'MB')

from cropcounter.train import load_checkpoint
from cropcounter.crop_dataset import CropTileDataset, load_records
from cropcounter.inference import predict_prob, decode_in_bounds
from cropcounter.metrics import evaluate
from cropcounter.dinov3_pyramid import amp_dtype
from torch.utils.data import DataLoader
from cropcounter.crop_dataset import collate_val

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_checkpoint(Path('weights/decoder_best.pt'), device, weights_dir=Path('weights'))
model.eval()
print('point-task decoder loaded strictly; task =', cfg.task)

recs = load_records(Path('examples/demo/data/val'), fmt='cvat')
ds = CropTileDataset(recs, Path('examples/demo/data/val/images'), train=False, output_stride=cfg.output_stride, sigma=cfg.sigma)
# Reference counts produced by this same decoder on MPS (fp32) on 2026-08-19, committed under examples/demo/.
expected = {r['image_name']: int(r['predicted_count']) for r in csv.DictReader(open('examples/demo/data/val/expected_counts.csv'))}
# Which numerics count as "the same model"? The committed reference was computed in fp32 on MPS.
# CUDA fp32 is not bit-identical to MPS fp32 (different cuDNN conv algorithms), and on Ampere
# PyTorch runs fp32 convolutions in TF32 by default. Observed on an A100: five images reproduce
# exactly, the densest (92 plants at tau=0.35) gives 89 in fp32 and 90 in bf16 -- a couple of
# threshold-straddling peaks, not a different backbone (a wrong backbone wrecks all six).
# So the asserted gate is fp32 with TF32 OFF at max(1, 5 %) per image; TF32 and the GPU's
# native AMP dtype (what training actually uses) are printed as information.
AMP = amp_dtype(device)
AMP_NAME = str(AMP).replace('torch.', '') if AMP else 'fp32'

def count(img, rec, amp):
    prob = predict_prob(model, img, device, amp=amp)
    pts, _ = decode_in_bounds(prob, rec.width, rec.height, tau=0.35, k=cfg.k, nms_radius=cfg.nms_radius, output_stride=cfg.output_stride)
    return len(pts)

tf32_defaults = (torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32)
ok = True
print(f'{"image":10s} {"fp32/noTF32":>12s} {"committed":>10s} {"Δ":>4s}  ||  {"fp32/TF32":>10s} {AMP_NAME:>9s}')
for i, rec in enumerate(recs):
    img = ds[i]['image']
    torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
    c_fp32 = count(img, rec, False)
    torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32 = tf32_defaults
    c_tf32 = count(img, rec, False)
    c_amp = count(img, rec, True)
    exp = expected[rec.name]
    ok &= abs(c_fp32 - exp) <= max(1, round(0.05 * exp))
    print(f'{rec.name:10s} {c_fp32:12d} {exp:10d} {c_fp32 - exp:+4d}  ||  {c_tf32:10d} {c_amp:9d}')
loader = DataLoader(ds, batch_size=1, collate_fn=collate_val)
summ, _ = evaluate(model, loader, device, tau=0.35, k=cfg.k, nms_radius=cfg.nms_radius, output_stride=cfg.output_stride, match_radius_px=cfg.match_radius_px)
print({k: round(v, 3) for k, v in summ.items()})
assert ok, 'converted backbone does NOT reproduce the committed wheat counts (fp32, TF32 off, 5 % gate) -- stop here'
print(f'backbone verified by fp32 metric reproduction ✓ | this GPU will train under {AMP_NAME}')


## 4 · Throughput probe — is the frozen path CPU-bound?

Measures images/s and GPU utilisation for a few `num_workers` settings on the real loader *before* any long run. If GPU utilisation sits well under ~60 % at the best worker count, do not buy a bigger GPU for the frozen runs — spend it on the unfrozen comparator (Protocol B/C).

In [ ]:
import subprocess, threading, statistics, torch, time
from cropcounter.train import TrainConfig, build_loaders, build_model, resolve_device
from cropcounter.dinov3_pyramid import autocast_context

base = TrainConfig.from_json('examples/FishDetection/config_8ep.json')
base.data_root = pathlib.Path(DATA); base.weights_dir = pathlib.Path('weights')
device = resolve_device(None)
model = build_model(base, device); model.train()

def gpu_util_sampler(stop, out):
    while not stop.is_set():
        try:
            u = subprocess.check_output(['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader,nounits']).decode().strip()
            out.append(int(u.split()[0]))
        except Exception:
            pass
        time.sleep(0.5)

results = {}
for nw in sorted({2, 4, min(8, os.cpu_count()), os.cpu_count()}):
    cfg = TrainConfig.from_dict(base.to_dict()); cfg.num_workers = nw
    train_loader, *_ = build_loaders(cfg, device)
    it = iter(train_loader)
    next(it)  # warm-up (worker spin-up)
    stop, samples = threading.Event(), []
    th = threading.Thread(target=gpu_util_sampler, args=(stop, samples)); th.start()
    n_img, t0 = 0, time.time()
    for _ in range(15):
        images, targets, _n = next(it)
        images = images.to(device, non_blocking=True)
        with autocast_context(device):
            out = model(images)
        torch.cuda.synchronize()
        n_img += images.shape[0]
    dt = time.time() - t0
    stop.set(); th.join()
    results[nw] = (n_img / dt, statistics.mean(samples) if samples else float('nan'))
    print(f'num_workers={nw:2d}: {n_img/dt:6.1f} img/s | GPU util {results[nw][1]:.0f}%')
    del it, train_loader
BEST_WORKERS = max(results, key=lambda k: results[k][0])
print('→ use num_workers =', BEST_WORKERS)
json.dump({str(k): v for k, v in results.items()}, open(f'{DRIVE}/results/throughput_probe.json', 'w'), indent=1)

# The training runs below are SEPARATE processes (`!python -m cropcounter.train`). Anything this
# kernel still holds on the GPU is taken away from them -- on an A100 the probe + backbone check
# left ~38 GB reserved in the kernel and the trainer OOM'd on its first forward. Release it.
import gc
for _name in ('model', 'out', 'images', 'targets', 'it', 'train_loader', 'loader', 'prob', 'ds'):
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f'kernel GPU memory after cleanup: reserved {torch.cuda.memory_reserved()/1e9:.2f} GB, allocated {torch.cuda.memory_allocated()/1e9:.2f} GB')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

## 5 · Train the frozen-trunk head on Brackish `is_train` (the go/no-go run)

Config = [`examples/FishDetection/config_8ep.json`](../config_8ep.json) — 8 epochs × 1 tile/frame, sized for a free-tier session — with `augment_profile: natural` (no vertical flips / 90° rotations underwater), log-space size parameterisation, `negative_tile_fraction 0.2` (the empty-image hazard is handled in the sampler, not the loss). Checkpoints, `history.json`, `curves.png` and `predictions.json` land on Drive as they are written.

In [ ]:
cfg = TrainConfig.from_json('examples/FishDetection/config_8ep.json')
cfg.data_root = pathlib.Path(DATA); cfg.weights_dir = pathlib.Path('weights')
cfg.out_dir = pathlib.Path(f'{DRIVE}/runs'); cfg.run_name = 'brackish_frozen_s0'
cfg.num_workers = BEST_WORKERS; cfg.seed = 0
# config_8ep.json already carries the trim: free-tier Colab caps a session at ~5 h, and
# 8 epochs x 1 tile/frame (~11.5k tiles/epoch on 960x540 frames) is enough for a go/no-go
# signal while leaving room for the linear probe + baseline in the same session.
cfg.to_json(pathlib.Path('/content/config_brackish_frozen.json'))
print(json.dumps(cfg.to_dict(), indent=1))

In [ ]:
# Guard: the trainer must get the GPU. If the kernel still holds it, restart the session
# (Runtime > Restart session) and run from the top -- the VM disk (frames, weights) survives.
import gc, torch
gc.collect(); torch.cuda.empty_cache()
_reserved_gb = torch.cuda.memory_reserved() / 1e9
print(f'kernel reserved {_reserved_gb:.2f} GB before launching the trainer')
assert _reserved_gb < 2.0, 'kernel is holding the GPU -- Runtime > Restart session, then Run all'
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
!python -m cropcounter.train --config /content/config_brackish_frozen.json 2>&1 | tee {DRIVE}/runs/brackish_frozen_s0.log

## 6 · Linear-probe diagnostic — freeze the fuse trunk too (2 epochs)

Non-trivial AP here ⇒ the DINOv3 features are *near-linearly* box-decodable and the thesis is strong. Near-zero while the full decoder works ⇒ the story is about the decoder, not the features — a materially different memo.

In [ ]:
# Guard: the trainer must get the GPU. If the kernel still holds it, restart the session
# (Runtime > Restart session) and run from the top -- the VM disk (frames, weights) survives.
import gc, torch
gc.collect(); torch.cuda.empty_cache()
_reserved_gb = torch.cuda.memory_reserved() / 1e9
print(f'kernel reserved {_reserved_gb:.2f} GB before launching the trainer')
assert _reserved_gb < 2.0, 'kernel is holding the GPU -- Runtime > Restart session, then Run all'
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
cfg_lp = TrainConfig.from_dict(cfg.to_dict())
cfg_lp.freeze_fusion = True; cfg_lp.epochs = 2; cfg_lp.warmup_epochs = 0; cfg_lp.run_name = 'brackish_linearprobe_s0'
cfg_lp.to_json(pathlib.Path('/content/config_brackish_linearprobe.json'))
!python -m cropcounter.train --config /content/config_brackish_linearprobe.json 2>&1 | tee {DRIVE}/runs/brackish_linearprobe_s0.log

## 7 · Next

`3_inference.ipynb` — the released RF-DETR baseline over the identical val images.